# Netflix EDA

This notebook performs exploratory data analysis on the Netflix titles dataset, including cleaning, date conversion, feature engineering, visualization, and a simple trend prediction step.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "netflix_titles.csv"
VISUALS_DIR = PROJECT_ROOT / "visuals"
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
df = df.drop_duplicates().copy()
df["director"] = df["director"].fillna("Unknown")
df["cast"] = df["cast"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")
df["rating"] = df["rating"].fillna("Unknown")
df["duration"] = df["duration"].fillna("Unknown")
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

df["content_type"] = df["type"].str.strip()
df["added_year"] = df["date_added"].dt.year
df["primary_genre"] = df["listed_in"].str.split(", ").str[0]
df["primary_country"] = df["country"].str.split(", ").str[0]
df["release_decade"] = (df["release_year"] // 10) * 10

df.info()

In [ ]:
df[["release_year", "added_year"]].describe()

In [ ]:
type_counts = df["content_type"].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(type_counts.values, labels=type_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Netflix Content Type Distribution")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "pie_content_type.png")
plt.show()

In [ ]:
yearly_trend = df.dropna(subset=["added_year"]).groupby("added_year").size()
plt.figure(figsize=(10, 5))
sns.lineplot(x=yearly_trend.index, y=yearly_trend.values, marker="o", color="teal")
plt.title("Yearly Trend of Titles Added to Netflix")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "yearly_trend.png")
plt.show()

In [ ]:
top_genres = df["listed_in"].str.split(", ").explode().value_counts().head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, hue=top_genres.index, palette="crest", legend=False)
plt.title("Top 10 Netflix Genres")
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "top_genres.png")
plt.show()

In [ ]:
country_type = (
    df[df["primary_country"] != "Unknown"]
    .groupby(["primary_country", "content_type"])
    .size()
    .unstack(fill_value=0)
)
top_countries = country_type.sum(axis=1).sort_values(ascending=False).head(10).index
top_country_type = country_type.loc[top_countries]
plt.figure(figsize=(10, 6))
sns.heatmap(top_country_type, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Top Countries by Content Type")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "country_heatmap.png")
plt.show()

In [ ]:
trend_series = yearly_trend[yearly_trend.index >= 2016]
coefficients = np.polyfit(trend_series.index.astype(int), trend_series.values, 1)
predicted_next_year = np.polyval(coefficients, trend_series.index.max() + 1)
print(f"Predicted titles for next year: {predicted_next_year:.0f}")